# 2 — Space Vector PWM (SVPWM)

> **Goal.** Derive SVPWM from the space-vector decomposition,
> recognize the **min-max common-mode injection** that produces the
> same result far more simply, verify the 15% output-voltage bonus,
> and run the open-loop switched simulation with SVPWM driving an
> LC + RL load.

**Prerequisites**

- `01_vsi_basics.ipynb` (this project)

**What you'll know at the end**

1. Where the **6 active vectors + 2 zero vectors** of SVPWM come
   from and how a target reference is decomposed into adjacent
   active vectors.
2. The closed-form **min-max injection** formula and why it produces
   the same line-to-line voltages as vector decomposition.
3. The **15% bonus**: peak line-to-line in linear SVPWM = $V_{dc}$;
   in SPWM = $V_{dc} \sqrt 3/2 \approx 0.866 V_{dc}$.
4. Spectral cleanliness: SVPWM packs all harmonics around multiples
   of $f_{sw}$, with the lowest-order harmonic at $f_{sw} - 2 f_o$
   (much higher than SPWM's $f_{sw} \pm 2 f_o$).


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

from vsi_3phase_model import (
    VSI3PhaseParams,
    spwm_duties, svpwm_duties,
    simulate_open_loop_svpwm,
    fundamental_rms, thd,
)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

p = VSI3PhaseParams()


## 1. The vector-decomposition view

The reference voltage at any instant is a vector
$\vec V_{ref}(t)$ in the αβ plane. In each $T_{sw}$, the inverter
applies a sequence of two adjacent active vectors (e.g. $V_1$ and
$V_2$) plus the zero vectors, weighted so that the **average** equals
$\vec V_{ref}$:

$$
T_{sw} \cdot \vec V_{ref} = T_1 \vec V_1 + T_2 \vec V_2 + T_0 \vec V_0
$$

With $|V_{1,2}| = \frac{2}{3} V_{dc}$ (Clarke-amplitude scaling),
and after solving the geometry:

$$
T_1 = T_{sw} \cdot \frac{|\vec V_{ref}|}{\frac{2}{3} V_{dc}} \cdot \sin(60° - \gamma)
$$

$$
T_2 = T_{sw} \cdot \frac{|\vec V_{ref}|}{\frac{2}{3} V_{dc}} \cdot \sin(\gamma)
$$

$$
T_0 = T_{sw} - T_1 - T_2
$$

where $\gamma$ is the angle within the sector. The **maximum
modulation index** is reached when $\vec V_{ref}$ hits the hexagon
inscribed circle, $|V_{ref}| = V_{dc}/\sqrt 3$ → peak phase voltage
$= V_{dc}/\sqrt 3$ → peak line-to-line $= V_{dc}$.

Compare SPWM: max phase voltage = $V_{dc}/2$ → peak line-to-line
$= V_{dc} \sqrt 3/2 = 0.866 V_{dc}$. SVPWM gives **15.5% more**.


## 2. The min-max injection shortcut

The "real" SVPWM implementation (sector identification, dwell times,
switching-pattern assembly) is conceptually clean but verbose. There's
a **theorem** (Holmes & Lipo §6.5): the same line-to-line voltages
are produced by simply **adding a common-mode offset** of
$-(\max + \min)/2$ to each phase reference before applying standard
SPWM:

$$
v_{offset} = -\frac{\max(v_a, v_b, v_c) + \min(v_a, v_b, v_c)}{2}
$$

$$
v_{x,pwm} = v_{x,ref} + v_{offset}
$$

$$
d_x = 0.5 + v_{x,pwm} / V_{dc}
$$

The common-mode injection shifts all three references by the same
amount — that's invisible at the load (which only sees phase-to-phase
voltages). Both implementations are equivalent line-to-line.

This is the implementation we use in `svpwm_duties()`. It's a
one-liner.


In [ ]:
# Plot: SPWM references vs SVPWM references (with injection)
t = np.linspace(0, 1.0/p.f_o, 1000)
V_ref_pk = p.V_o_LN_pk
v_a = V_ref_pk * np.cos(p.omega_o * t)
v_b = V_ref_pk * np.cos(p.omega_o * t - 2*np.pi/3)
v_c = V_ref_pk * np.cos(p.omega_o * t + 2*np.pi/3)
v_max = np.maximum(np.maximum(v_a, v_b), v_c)
v_min = np.minimum(np.minimum(v_a, v_b), v_c)
v_offset = -(v_max + v_min) / 2.0
v_a_inj = v_a + v_offset

fig, axs = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axs[0].plot(t*1000, v_a, "C0", label="$v_{ref,a}$ (sinusoid)")
axs[0].plot(t*1000, v_a_inj, "C3", linewidth=2, label="$v_{ref,a} + v_{offset}$ (SVPWM)")
axs[0].plot(t*1000, v_offset, "C2--", alpha=0.5, label="$v_{offset}$ (common-mode)")
axs[0].set_ylabel("Voltage [V]"); axs[0].legend(loc="lower right")
axs[0].set_title("Min-max injection: 3rd-harmonic-like common-mode shifts phase refs")

# Compare duties
d_a_spwm, _, _ = spwm_duties(v_a, v_b, v_c, p.V_dc)
d_a_svpwm, _, _ = svpwm_duties(v_a, v_b, v_c, p.V_dc)
axs[1].plot(t*1000, d_a_spwm, "C0", label="$d_a$ SPWM")
axs[1].plot(t*1000, d_a_svpwm, "C3", linewidth=2, label="$d_a$ SVPWM (via injection)")
axs[1].axhline(0.5, color="k", linestyle=":", alpha=0.3)
axs[1].set_ylabel("Duty"); axs[1].set_xlabel("Time [ms]")
axs[1].legend(loc="lower right")
plt.tight_layout(); plt.show()
print(f"SPWM  duty range: [{d_a_spwm.min():.4f}, {d_a_spwm.max():.4f}]")
print(f"SVPWM duty range: [{d_a_svpwm.min():.4f}, {d_a_svpwm.max():.4f}]")
print(f"SVPWM lets us reach m_a = {p.m_a:.3f} (would saturate at SPWM)")


## 3. Open-loop SVPWM switched simulation

Run the full switched simulator with SVPWM. The line-to-line voltage
at the load (post-filter) should be a clean 230 V_rms sinusoid with
low THD.


In [ ]:
sim = simulate_open_loop_svpwm(p, m_a=p.m_a, n_cycles=3, samples_per_period=80,
                                use_svpwm=True)
mask = sim['t'] > 1.0/p.f_o
fs = 1.0/(sim['t'][1] - sim['t'][0])

# Post-filter line-to-line voltage
v_LL_post_filter = sim['v_Ca'] - sim['v_Cb']
v_LL_fund = fundamental_rms(v_LL_post_filter[mask], fs, p.f_o)
v_LL_thd = thd(v_LL_post_filter[mask], fs, p.f_o, 30)

fig, axs = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
axs[0].plot(sim['t']*1000, sim['v_pole_a'], "C0", linewidth=0.5, alpha=0.7)
axs[0].set_ylabel("$v_{pole,a}$ [V]"); axs[0].set_title("SVPWM switched waveforms")

axs[1].plot(sim['t']*1000, sim['v_LL_ab'], "C3", linewidth=0.3, alpha=0.5,
            label="Pre-filter $v_{LL,ab}$")
axs[1].set_ylabel("Line-to-line [V]"); axs[1].legend(loc="upper right")

axs[2].plot(sim['t']*1000, v_LL_post_filter, "C2", linewidth=1.2,
            label="Post-filter $v_{Ca}-v_{Cb}$")
axs[2].set_ylabel("Filtered LL [V]"); axs[2].set_xlabel("Time [ms]")
axs[2].legend(loc="upper right")
plt.tight_layout(); plt.show()

print(f"Post-filter LL fundamental rms = {v_LL_fund:.2f} V  (target {p.V_o_LL_rms:.2f} V)")
print(f"Post-filter LL THD             = {v_LL_thd*100:.2f} %  (target < 3% for clean output)")


## 4. Spectrum: SPWM vs SVPWM

Both SVPWM and SPWM produce harmonic content centered around $f_{sw}$
and its multiples. The two differ in how the harmonics are
distributed and which ones cancel out. SVPWM produces somewhat lower
distortion at the same $m_a$ — and crucially supports a higher $m_a$
ceiling (1.0 vs 0.866).


In [ ]:
# Run both at the same m_a (where both work) and compare spectra
m_a_compare = 0.8  # below SPWM ceiling

sim_spwm = simulate_open_loop_svpwm(p, m_a=m_a_compare, n_cycles=4,
                                      samples_per_period=80, use_svpwm=False)
sim_svpwm = simulate_open_loop_svpwm(p, m_a=m_a_compare, n_cycles=4,
                                       samples_per_period=80, use_svpwm=True)

# Compute FFTs of the pre-filter line-to-line voltages
fs = 1.0/(sim_spwm['t'][1] - sim_spwm['t'][0])
mask_steady = sim_spwm['t'] > 1.0/p.f_o
for name, sim in [("SPWM", sim_spwm), ("SVPWM", sim_svpwm)]:
    x = sim['v_LL_ab'][mask_steady]
    X = np.fft.rfft(x)
    freqs = np.fft.rfftfreq(len(x), d=1.0/fs)
    mag = np.abs(X) * 2.0 / len(x)
    # Normalize to fundamental
    idx_fund = np.argmin(np.abs(freqs - p.f_o))
    fund_mag = mag[idx_fund]
    plt.semilogy(freqs/1000, mag/fund_mag, label=name, alpha=0.7, linewidth=0.6)

plt.axvline(p.f_sw/1000, color="C3", linestyle=":", alpha=0.5,
            label=f"$f_{{sw}}$ = {p.f_sw/1000:.0f} kHz")
plt.axvline(2*p.f_sw/1000, color="C3", linestyle=":", alpha=0.5)
plt.xlabel("Frequency [kHz]"); plt.ylabel("Magnitude (norm. to fundamental)")
plt.title(f"Spectrum of pre-filter line-to-line voltage @ $m_a$ = {m_a_compare}")
plt.legend(); plt.xlim(0, 60); plt.ylim(1e-4, 2)
plt.tight_layout(); plt.show()

# Compute THDs
thd_spwm = thd(sim_spwm['v_LL_ab'][mask_steady], fs, p.f_o, 40)
thd_svpwm = thd(sim_svpwm['v_LL_ab'][mask_steady], fs, p.f_o, 40)
print(f"Pre-filter LL THD @ m_a={m_a_compare}:  SPWM = {thd_spwm*100:.1f}%,  "
      f"SVPWM = {thd_svpwm*100:.1f}%")


## 5. Summary

SVPWM is a small algorithmic upgrade with a big output payoff:

- **15.5% more output** at the same DC bus, same load, same filter
- Cleaner spectrum (slightly lower THD at the same $m_a$, and a
  higher achievable $m_a$ ceiling)
- Same implementation cost via **min-max injection** — adding 5
  lines to a sinusoidal PWM modulator

Production motor drives, EV inverters, and grid-tie converters all
use SVPWM (or a close variant like DPWM that further reduces
switching losses). It's the standard.

**Next**: `03_vsi_standalone.ipynb` — close the voltage loop in dq
and drive an RL load with regulation.
